In [7]:
# # 🏏 IPL Match Winner Predictor
# Predicts the likely winner between two IPL teams given the **venue**, **toss**, and **pitch conditions** --
# using dropdown menus.



# 1. Upload the CSV files
from google.colab import files
uploaded = files.upload()   # select ipl_matches.csv AND stadium_pitch_conditions.csv here

# 2. Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 3. Load data
matches = pd.read_csv("ipl_matches.csv")
stadium = pd.read_csv("stadium_pitch_conditions.csv")

print(matches.shape, stadium.shape)
matches.head()

# 4. Merge stadium/pitch reference info into the match data
df = matches.merge(
    stadium[["venue", "avg_first_innings_score", "spin_friendliness", "pace_friendliness"]],
    on="venue", how="left"
)
df.head()

# 5. Feature engineering: historical win rate per team
# (Raw team names alone carry no numeric meaning for a model, so we derive
#  each team's overall win rate from the match history -- this is what
#  actually lets the model learn "who's the stronger side".)
win_counts = matches["winner"].value_counts()
played_counts = pd.concat([matches["team1"], matches["team2"]]).value_counts()
team_win_rate = (win_counts / played_counts).fillna(0.5)

df["team1_win_rate"] = df["team1"].map(team_win_rate)
df["team2_win_rate"] = df["team2"].map(team_win_rate)

team_win_rate.sort_values(ascending=False)

# 6. Encode categorical columns
cat_cols = ["team1", "team2", "venue", "pitch_type", "toss_winner", "toss_decision"]
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col])
    encoders[col] = le

# Target: did team1 win? (1 = team1 won, 0 = team2 won)
df["team1_won"] = (df["winner"] == df["team1"]).astype(int)

feature_cols = [c + "_enc" for c in cat_cols] + [
    "avg_first_innings_score", "spin_friendliness", "pace_friendliness",
    "team1_win_rate", "team2_win_rate",
]

X = df[feature_cols]
y = df["team1_won"]

# 7. Train / test split + model training
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=300, max_depth=8, random_state=42, class_weight="balanced"
)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

# 8. Feature importance (which factors matter most)
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances

# ## 9. Prediction Function
# This defines the core prediction logic. Run this cell once, then use the dropdown UI below it.

def predict_winner(team1, team2, venue, pitch_type, toss_winner, toss_decision):
    row = {
        "team1": team1,
        "team2": team2,
        "venue": venue,
        "pitch_type": pitch_type,
        "toss_winner": toss_winner,
        "toss_decision": toss_decision,
    }

    # pull venue stats, fall back to dataset averages if venue not found
    venue_row = stadium[stadium["venue"] == venue]
    if len(venue_row) == 0:
        avg_score = stadium["avg_first_innings_score"].mean()
        spin_f = stadium["spin_friendliness"].mean()
        pace_f = stadium["pace_friendliness"].mean()
    else:
        avg_score = venue_row["avg_first_innings_score"].values[0]
        spin_f = venue_row["spin_friendliness"].values[0]
        pace_f = venue_row["pace_friendliness"].values[0]

    encoded = {}
    for col in cat_cols:
        le = encoders[col]
        val = row[col]
        if val in le.classes_:
            encoded[col + "_enc"] = le.transform([val])[0]
        else:
            # unseen category -> use the most frequent class as a safe fallback
            encoded[col + "_enc"] = le.transform([le.classes_[0]])[0]

    t1_wr = team_win_rate.get(team1, 0.5)
    t2_wr = team_win_rate.get(team2, 0.5)

    input_df = pd.DataFrame([{
        **encoded,
        "avg_first_innings_score": avg_score,
        "spin_friendliness": spin_f,
        "pace_friendliness": pace_f,
        "team1_win_rate": t1_wr,
        "team2_win_rate": t2_wr,
    }])[feature_cols]

    prob_team1 = model.predict_proba(input_df)[0][1]
    prob_team2 = 1 - prob_team1

    winner = team1 if prob_team1 >= 0.5 else team2

    print(f"{team1} win probability: {prob_team1*100:.1f}%")
    print(f"{team2} win probability: {prob_team2*100:.1f}%")
    print(f"\nPredicted winner: {winner}")

    # simple probability bar chart
    fig, ax = plt.subplots(figsize=(5, 2.5))
    bars = ax.barh([team2, team1], [prob_team2, prob_team1], color=["#888888", "#1f77b4"])
    ax.set_xlim(0, 1)
    ax.set_xlabel("Win probability")
    ax.set_title(f"{team1} vs {team2} @ {venue}")
    for bar, p in zip(bars, [prob_team2, prob_team1]):
        ax.text(p + 0.02, bar.get_y() + bar.get_height() / 2, f"{p*100:.1f}%", va="center")
    plt.tight_layout()
    plt.show()

    return winner, prob_team1, prob_team2

# ## 10. (Optional) List available values
# Quick reference for exact spellings used in the dropdowns below.

print("Teams:", sorted(set(matches['team1']) | set(matches['team2'])))
print("\nVenues:", sorted(matches['venue'].unique()))
print("\nPitch types:", sorted(matches['pitch_type'].unique()))
print("\nToss decisions:", sorted(matches['toss_decision'].unique()))

# ## 11. Interactive Predictor (dropdowns)
# Pick **Team 1**, **Team 2**, the **ground**, who **won the toss**, and what they chose to do,
# then click **Predict Winner**. The pitch type auto-fills from the ground (you can still override it).

team_list = sorted(set(matches["team1"]) | set(matches["team2"]))
venue_list = sorted(matches["venue"].unique())
pitch_list = sorted(matches["pitch_type"].unique())
toss_decision_list = sorted(matches["toss_decision"].unique())

team1_dd = widgets.Dropdown(options=team_list, value=team_list[0], description="Team 1:")
team2_dd = widgets.Dropdown(options=team_list, value=team_list[1], description="Team 2:")
venue_dd = widgets.Dropdown(options=venue_list, value=venue_list[0], description="Ground:")
pitch_dd = widgets.Dropdown(options=pitch_list, description="Pitch type:")
toss_winner_dd = widgets.Dropdown(options=[team1_dd.value, team2_dd.value], description="Toss won by:")
toss_decision_dd = widgets.Dropdown(options=toss_decision_list, description="Toss choice:")
predict_btn = widgets.Button(description="Predict Winner", button_style="success", icon="play")
output_area = widgets.Output()


def refresh_toss_winner_options(change=None):
    current = toss_winner_dd.value
    toss_winner_dd.options = [team1_dd.value, team2_dd.value]
    if current not in toss_winner_dd.options:
        toss_winner_dd.value = team1_dd.value


def auto_fill_pitch(change=None):
    row = stadium[stadium["venue"] == venue_dd.value]
    if len(row):
        pitch_dd.value = row["pitch_type"].values[0]


team1_dd.observe(refresh_toss_winner_options, names="value")
team2_dd.observe(refresh_toss_winner_options, names="value")
venue_dd.observe(auto_fill_pitch, names="value")
auto_fill_pitch()  # set initial pitch value for the default ground


def on_predict_click(b):
    with output_area:
        clear_output(wait=True)
        if team1_dd.value == team2_dd.value:
            print("Please choose two different teams.")
            return
        predict_winner(
            team1=team1_dd.value,
            team2=team2_dd.value,
            venue=venue_dd.value,
            pitch_type=pitch_dd.value,
            toss_winner=toss_winner_dd.value,
            toss_decision=toss_decision_dd.value,
        )


predict_btn.on_click(on_predict_click)

ui = widgets.VBox([
    widgets.HBox([team1_dd, team2_dd]),
    widgets.HBox([venue_dd, pitch_dd]),
    widgets.HBox([toss_winner_dd, toss_decision_dd]),
    predict_btn,
    output_area,
])
display(ui)

Saving ipl_matches (1).csv to ipl_matches (1) (3).csv
Saving stadium_pitch_conditions (1).csv to stadium_pitch_conditions (1) (3).csv
(2500, 9) (10, 6)
Accuracy: 0.652
              precision    recall  f1-score   support

           0       0.63      0.68      0.66       245
           1       0.67      0.62      0.65       255

    accuracy                           0.65       500
   macro avg       0.65      0.65      0.65       500
weighted avg       0.65      0.65      0.65       500

Teams: ['Chennai Super Kings', 'Delhi Capitals', 'Gujarat Titans', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Punjab Kings', 'Rajasthan Royals', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']

Venues: ['Arun Jaitley Stadium, Delhi', 'Eden Gardens, Kolkata', 'Ekana Stadium, Lucknow', 'M. Chinnaswamy Stadium, Bengaluru', 'MA Chidambaram Stadium, Chennai', 'Narendra Modi Stadium, Ahmedabad', 'PCA Stadium, Mullanpur', 'Rajiv Gandhi Intl Stadium, Hyderabad', 'Sawai Mansingh